In [ ]:
import os
from fitparse import FitFile

193

In [122]:
from fitparse import FitFile

files = [file for file in os.listdir(os.path.join("inputs/activities/2025")) if file.endswith(".fit")]

fitfile = FitFile(os.path.join("inputs/activities/2025", files[4]))

for activity in fitfile.get_messages("sport"):
    for field in activity:
        print(field.name, field.value)

for index, record in enumerate(fitfile.get_messages("record")):
    data = {}
    data[index] = {field.name: field.value for field in record}

    # print(data)

records = [[(field.name, field.value) for field in record] for record in fitfile.get_messages("record")]
records
# records = {record["timestamp"]: {field.name: field.value for field in record} for record in fitfile.get_messages("record")}

name Treadmill
sport running
sub_sport treadmill
unknown_10 (253, 0, 0)
unknown_12 None
unknown_5 1
unknown_6 0


[[('cadence', 0),
  ('distance', 0.0),
  ('enhanced_speed', 0.0),
  ('fractional_cadence', 0.0),
  ('heart_rate', 122),
  ('timestamp', datetime.datetime(2025, 1, 24, 19, 30)),
  ('unknown_134', None),
  ('unknown_135', 16),
  ('unknown_136', 122),
  ('unknown_143', 22),
  ('unknown_87', 0)],
 [('cadence', 0),
  ('distance', 0.0),
  ('enhanced_speed', 0.0),
  ('fractional_cadence', 0.0),
  ('heart_rate', 123),
  ('timestamp', datetime.datetime(2025, 1, 24, 19, 30, 1)),
  ('unknown_134', None),
  ('unknown_135', 8),
  ('unknown_136', 123),
  ('unknown_143', 22)],
 [('cadence', 60),
  ('distance', 1.72),
  ('enhanced_speed', 1.39),
  ('fractional_cadence', 0.0),
  ('heart_rate', 126),
  ('timestamp', datetime.datetime(2025, 1, 24, 19, 30, 6)),
  ('unknown_134', None),
  ('unknown_135', 1),
  ('unknown_136', 126),
  ('unknown_143', 22)],
 [('cadence', 77),
  ('distance', 17.76),
  ('enhanced_speed', 2.501),
  ('fractional_cadence', 0.0),
  ('heart_rate', 129),
  ('timestamp', datetime.dat

In [96]:
files[3]

'2025-01-23-18_41-121607630.fit'

In [ ]:
class Parser:
    def __init__(self, directory="inputs/activities/2025", filetype=".fit"):
        self.directory = directory
        self.filetype = filetype
        self.files = self.list_files_in_directory()

    def list_files_in_directory(self):
        """List all files in the given directory."""
        try:
            files = os.listdir(self.directory)
            return [f for f in files if os.path.isfile(os.path.join(self.directory, f)) and f.endswith(self.filetype)]
        except FileNotFoundError:
            return f"The directory {self.directory} does not exist."
        except PermissionError:
            return f"Permission denied to access the directory {self.directory}."
    
    def get_data(self):
        if not hasattr(self, 'data'):
            self.data = self.parse_files()
        return self.data
    
    def parse_files(self):
        data = {file : {} for file in files}

        counter = 0
        for file in files:
            counter += 1
            print(f"Processing file {counter} of {len(files)}: {file}")
            fitfile = FitFile(os.path.join("inputs/activities/2025", file))
            
            sport = {}
            for activity in fitfile.get_messages("sport"):
                sport[activity.name] = {field.name: field.value for field in activity}
            records = []
            for record in fitfile.get_messages("record"):
                records.append({field["timestamp"]: {field.name: field.value} for field in record})
            data[file] = {"sport": sport, "records": records}
            # try:
            #     for index, record in enumerate(fitfile.get_messages("record")):
            #         data[file][index] = {field.name: field.value for field in record}
            # except Exception as e:
            #     print(f"Error processing records in file {file}: {e}")
        return data

In [124]:
parser = Parser()

In [125]:
data = parser.get_data()

data


Processing file 1 of 193: 2025-01-20-20_43-121607623.fit


AttributeError: 'FieldData' object has no attribute 'timestamp'

In [ ]:
parsed_data = {}

for entry_name, item in data.items():
    try:
        date = item.get("record", {}).get("timestamp").date()
    except AttributeError:
        print(f"impossible to parse date from {entry_name}", item)
        date = None
    sport = item.get("sport", {}).get("sport")
    subsport = item.get("sport", {}).get("sub_sport", "unknown")
    parsed_data[entry_name] = {
        "date": date,
        "sport": sport,
        "subsport": subsport
    }

In [101]:
import pandas as pd
df = pd.DataFrame.from_dict(parsed_data, orient="index")

In [104]:
df_grouped = df.groupby("date").size()
df_grouped

Series([], dtype: int64)

In [103]:
import plotly.express as px

px.bar(df_grouped, labels={"index": "Date", "value": "Number of Activities"}, title="Activities per Day")

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [],
    'layout': {'barmode': 'relative',
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Activities per Day'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'date'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'Number of Activities'}}}
})